# Phase 9 — DistilBERT Train+Valid Fine-tuning

Εκπαίδευση DistilBERT σε **train+valid μαζί** με σταθερά 9 epochs.

**Γιατί αυτή η στρατηγική:**
- Το best Kaggle score (0.755) ήρθε από ensemble DistilBERT+SVM
- Το DistilBERT είχε validation 0.7998 αλλά Kaggle 0.723 → overfitting στο validation
- Εκπαιδεύοντας σε train+valid (5647 δείγματα αντί 5082) το μοντέλο γενικεύει καλύτερα
- Σταθερά 9 epochs (βέλτιστο από Phase 2) χωρίς early stopping

**Αναμενόμενο αποτέλεσμα:** Καλύτερο Kaggle score από 0.755

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import copy
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Συνδυασμός train + valid
train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

print(f'Train (original): {len(train)}')
print(f'Valid (original): {len(valid)}')
print(f'Train+Valid:      {len(train_full)}  ← εκπαίδευση σε αυτό')
print(f'Test:             {len(texts_test)}')

In [ ]:
# Label Encoding — fit σε train+valid για να δει όλες τις κλάσεις
le_hazard  = LabelEncoder()
le_product = LabelEncoder()

le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_full_product = le_product.transform(train_full['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Hazard classes:  {n_hazard}')
print(f'Product classes: {n_product}')

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
BATCH_SIZE = 32
MAX_LENGTH = 128
LR         = 2e-5
N_EPOCHS   = 9      # Σταθερά 9 epochs — βέλτιστο από Phase 2
WARMUP_RATIO = 0.1

print(f'Φόρτωση tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded!')

dummy_labels = np.zeros(len(texts_test), dtype=int)

In [ ]:
class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)


def get_predictions(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = model(**batch).logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

In [ ]:
# DataLoaders — μόνο train (full) και test, χωρίς validation
full_loader_hazard  = DataLoader(FoodHazardDataset(texts_full, y_full_hazard,  tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
full_loader_product = DataLoader(FoodHazardDataset(texts_full, y_full_product, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
test_loader_hazard  = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader_product = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

print(f'Full train batches: {len(full_loader_hazard)}')
print(f'Batch size: {BATCH_SIZE} | Epochs: {N_EPOCHS} | LR: {LR}')

## Εκπαίδευση DistilBERT — Hazard Classifier
Εκπαίδευση σε train+valid, σταθερά 9 epochs, χωρίς early stopping.

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ DistilBERT (train+valid) — HAZARD ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS}\n')

bert_hazard = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_hazard
).to(device)

optimizer_hazard = AdamW(bert_hazard.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(full_loader_hazard) * N_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler_hazard = get_linear_schedule_with_warmup(
    optimizer_hazard,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_hazard, full_loader_hazard, optimizer_hazard, scheduler_hazard)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

torch.save(bert_hazard.state_dict(), 'bert_fulldata_hazard.pt')
print(f'\n Αποθηκεύτηκε: bert_fulldata_hazard.pt')

## Εκπαίδευση DistilBERT — Product Classifier

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ DistilBERT (train+valid) — PRODUCT ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS}\n')

bert_product = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=n_product
).to(device)

optimizer_product = AdamW(bert_product.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(full_loader_product) * N_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler_product = get_linear_schedule_with_warmup(
    optimizer_product,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(bert_product, full_loader_product, optimizer_product, scheduler_product)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {loss:.4f}')

torch.save(bert_product.state_dict(), 'bert_fulldata_product.pt')
print(f'\n Αποθηκεύτηκε: bert_fulldata_product.pt')

## Αποθήκευση Probabilities για Ensemble με SVM

In [ ]:
# Test probabilities για ensemble
bert_test_hazard_probs  = get_probabilities(bert_hazard,  test_loader_hazard)
bert_test_product_probs = get_probabilities(bert_product, test_loader_product)

np.save('bert_fulldata_test_hazard_probs.npy',  bert_test_hazard_probs)
np.save('bert_fulldata_test_product_probs.npy', bert_test_product_probs)

print(f'Test hazard probs shape:  {bert_test_hazard_probs.shape}')
print(f'Test product probs shape: {bert_test_product_probs.shape}')
print('\nΑποθηκεύτηκαν τα .npy για ensemble')

## Submission — DistilBERT μόνο (train+valid)

In [ ]:
# Test predictions
test_hazard  = le_hazard.inverse_transform(get_predictions(bert_hazard,  test_loader_hazard))
test_product = le_product.inverse_transform(get_predictions(bert_product, test_loader_product))

submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
})
submission.to_csv('submission_bert_fulldata.csv', index=False)
print('Αποθηκεύτηκε: submission_bert_fulldata.csv')
print(f'Predictions: {len(submission)}')
print('\n--- Πρώτες 5 ---')
print(submission.head())
